In [6]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

def get_weight(isbn):
    """
    기존 쪽수 추출 예제를 응용하여 도서의 무게(weight_g)를 추가로 추출하고 DataFrame에 저장
    """
    search_url = f"http://www.yes24.com/Product/Search?domain=BOOK&query={isbn}"

    r = requests.get(search_url, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")

    anchor = soup.find("a", class_="gd_name")

    if anchor is None:
        return ""

    detail_url = "http://www.yes24.com" + anchor["href"]

    r = requests.get(detail_url, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")

    for tr in soup.select("#infoset_specific tr"):

        th = tr.find("th")

        if th and th.get_text(strip=True) == "쪽수, 무게, 크기":

            raw = tr.find("td").get_text(" ", strip=True)

            tokens = re.split(r"[|,\s]+", raw)

            weight_g = next(
                (t.replace("g", "") for t in tokens if "g" in t),
                ""
            )

            return weight_g

    return ""

In [7]:
top10_books["weight_g"] = top10_books["isbn13"].apply(get_weight)

top10_books.head()

NameError: name 'top10_books' is not defined

In [8]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

def get_page_and_weight(isbn):
    """
    기존 함수를 확장하여 쪽수(page_cnt)와 무게(weight_g)를 동시에 추출하고 두 개의 컬럼으로 저장
    """
    search_url = f"http://www.yes24.com/Product/Search?domain=BOOK&query={isbn}"

    r = requests.get(search_url, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")

    anchor = soup.find("a", class_="gd_name")

    if anchor is None:
        return "", ""

    detail_url = "http://www.yes24.com" + anchor["href"]

    r = requests.get(detail_url, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")

    for tr in soup.select("#infoset_specific tr"):

        th = tr.find("th")

        if th and th.get_text(strip=True) == "쪽수, 무게, 크기":

            raw = tr.find("td").get_text(" ", strip=True)

            tokens = re.split(r"[|,\s]+", raw)

            page_cnt = next(
                (t.replace("쪽", "") for t in tokens if "쪽" in t),
                ""
            )

            weight_g = next(
                (t.replace("g", "") for t in tokens if "g" in t),
                ""
            )

            return page_cnt, weight_g

    return "", ""

In [9]:
top10_books[["page_cnt", "weight_g"]] = (
    top10_books["isbn13"]
    .apply(lambda isbn: pd.Series(get_page_cnt(isbn)))
)

top10_books.head()

NameError: name 'top10_books' is not defined